# Reset Kernel
Uncomment and run cell to reset kernel

In [ ]:
# import IPython
# IPython.Application.instance().kernel.do_shutdown(True)

# Main Kaggle deploy

In [ ]:
!pip install vllm==0.10.0
!pip install triton==3.2.0
!pip install langchain langchain-community langchain_google_genai openai faiss-cpu langchain_huggingface crawl4ai unidecode pymupdf4llm google-genai
!pip uninstall -y openai
!pip install openai==1.90.0

In [ ]:
# Install pyngrok
!pip install pyngrok

# Import and setup ngrok
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
import time
import sys

user_secrets = UserSecretsClient()
NGROK_KEY = user_secrets.get_secret("NGROK_KEY")

# Set ngrok authtoken
ngrok.set_auth_token(NGROK_KEY)

In [ ]:
import os
BRAVE_API_KEY = user_secrets.get_secret("BRAVE_API_KEY")
GOOGLE_API_KEY = user_secrets.get_secret("GOOGLE_API_KEY")

os.environ["WEB_SEARCH_SSL"] = str(False)
os.environ["GOOGLE_SEARCH_CX"] = "9501a956284f141ab"
os.environ["GOOGLE_SEARCH_API_KEY"] = GOOGLE_API_KEY
os.environ["BRAVE_SEARCH_API_KEY"] = BRAVE_API_KEY

DOMAIN = "http://13.215.102.203:8000"
NGROK_PORT = 8002

In [ ]:
# Start ngrok tunnel using pyngrok
tunnel = ngrok.connect(NGROK_PORT, "http")
public_url = tunnel.public_url

# Sử dụng ngrok URL làm server ID thay vì random UUID
SERVER_ID = public_url.replace("https://", "").replace("http://", "")

print(f"Initializing Kaggle Worker: {SERVER_ID}")
print(f"Ngrok tunnel started!")

In [ ]:
import requests
import io
import tarfile
import shutil

def wait_for_approval(server_id: str, max_wait_minutes: int = 5):
    """Chờ admin approve request"""
    print(f"Waiting for admin approval for server {server_id}...")
    
    start_time = time.time()
    check_interval = 2
    contact_shown = False
    
    while time.time() - start_time < max_wait_minutes * 60:
        try:
            response = requests.get(f"{DOMAIN}/admin/kaggle/check-status/{server_id}", timeout=10)
            if response.status_code == 200:
                data = response.json()
                if data.get("success"):
                    status = data.get("next", {}).get("status")
                    if status == "approved":
                        print("Server approved!")
                        admin_notes = data.get("next", {}).get("admin_notes")
                        approved_by = data.get("next", {}).get("approved_by")
                        approved_at = data.get("next", {}).get("approved_at")
                        print(f"Admin notes: {admin_notes}")
                        print(f"Approved by: {approved_by}")
                        print(f"Approved at: {approved_at}")
                        return data.get("next")
                    elif status == "rejected":
                        print("Server request was rejected.")
                        admin_notes = data.get("next", {}).get("admin_notes")
                        rejected_by = data.get("next", {}).get("rejected_by")
                        rejected_at = data.get("next", {}).get("rejected_at")
                        print(f"Admin notes: {admin_notes}")
                        print(f"Rejected by: {rejected_by}")
                        print(f"Rejected at: {rejected_at}")
                        sys.exit(1)
                    else:
                        # Hiển thị contact admin info
                        if not contact_shown:
                            next_data = data.get("next", {})
                            contact_admin = next_data.get("contact_admin")
                            message = next_data.get("message")
                            if contact_admin:
                                print(f"Contact admin for approval: {contact_admin}")
                            if message:
                                print(f"{message}")
                            contact_shown = True
            
            time.sleep(check_interval)
            
        except Exception as e:
            print(f"Error checking approval status: {e}")
            time.sleep(check_interval)
    
    print("Timeout after 1 minute. Please try again later or contact admin.")
    sys.exit(1)

def request_server_access():
    """Request access từ admin"""
    try:
        data = {
            "server_id": SERVER_ID,
            "request_type": "server_registration",
            "requested_packages": ["vllm_worker", "web_search", "server"],
            "reason": f"Kaggle deployment for UniAdmission ChatBot. Server URL: {public_url}",
            "contact_info": "kaggle-worker@uniadmission.ai",
            "ngrok_url": public_url
        }

        response = requests.post(f"{DOMAIN}/admin/kaggle/request-access", json=data, timeout=10)
                
        if response.status_code == 200:
            try:
                result = response.json()
                
                if result.get("success"):
                    print("Access request submitted successfully")
                    return True
                else:
                    print(f"Failed to request access: {result.get('detail', 'Unknown error')}")
                    return False
            except Exception as json_error:
                print(f"Error parsing JSON response: {json_error}")
                return False
        else:
            print(f"Failed to request access: {response.status_code} - {response.text}")
            return False
            
    except Exception as e:
        print(f"Error requesting access: {e}")
        return False


def check_already_approved():
    try:
        response = requests.get(f"{DOMAIN}/admin/kaggle/check-status/{SERVER_ID}", timeout=10)
        if response.status_code == 200:
            data = response.json()
            return data.get("success") and data.get("next", {}).get("status") == "approved"
    except:
        pass
    return False

if not check_already_approved():
    print("Requesting server access...")
    request_id = request_server_access()
    if not request_id:
        print("Failed to request access. Stopping.")
        sys.exit(1)
    wait_for_approval(SERVER_ID)
else:
    print("Server already approved!")

In [ ]:
def unpack(data: bytes, path: str):
    if os.path.exists(path):
        shutil.rmtree(path)
    with io.BytesIO(data) as tar_buffer:
        with tarfile.open(fileobj=tar_buffer, mode='r:gz') as tar:
            tar.extractall(path=path)

def unpack_p(name: str):
    """Download package với server_id parameter"""
    response = requests.get(f"{DOMAIN}/script/{name}?server_id={SERVER_ID}", timeout=30)
    
    if response.status_code == 200:
        unpack(response.content, name)
        print(f"Successfully downloaded and unpacked: {name}")
    elif response.status_code == 403:
        print(f"Access denied for package {name}. Server may not be approved.")
        sys.exit(1)
    else:
        print(f"Failed to download {name}: {response.status_code} - {response.text}")
        sys.exit(1)

def unpack_pl(*names: str):
    for name in names:
        unpack_p(name)

# Kiểm tra và xin access approval
print(f"Checking server {SERVER_ID} approval status...")

# Tải packages sau khi đã được approve
unpack_pl("vllm_worker", "web_search", "server")

In [ ]:
from vllm_worker import prepare
from server import construct_app, CustomQA, create_client_info, set_active, PRELOAD_MODEL, KaggleRequest
from typing import AsyncGenerator
import uvicorn
prepare()
CLIENT_INFO = create_client_info()

In [ ]:
import threading
def thread_task():
    ws_pipeline = CustomQA(set_active)
    REQUEST_STORAGE: dict[str, tuple[str, KaggleRequest]] = {}
    
    async def pre_inference_function(request: KaggleRequest):
        prompt, pre_output = await ws_pipeline.pre_inference(
            request["model_id"],
            request["question"],
            request["stream_id"],
            request["params"],
            request.get("vector_sources"),
            request.get("web_search_keywords")
        )
        REQUEST_STORAGE[request["stream_id"]] = (prompt, request)
        return pre_output
    
    async def inference_function(stream_id: str) -> AsyncGenerator[str, None]:
        prompt, request = REQUEST_STORAGE.pop(stream_id)
        generator = await ws_pipeline.inference(prompt, request)
        return generator
    
    app = construct_app(
        server_domain=DOMAIN,
        info=CLIENT_INFO,
        pre_inference=pre_inference_function,
        inferece=inference_function,
        init_tasks=[
            ws_pipeline.start(),
            ws_pipeline.preload(PRELOAD_MODEL)
        ]
    )
    
    uvicorn.run(app, port=NGROK_PORT)

thread = threading.Thread(target=thread_task, daemon=True)
thread.start()